> Pedro Juan Torres González (z32togop@uco.es)

> José Checa Claudel (https://www.uco.es/atdfiware/)

> https://github.com/P3J0T4TG/Hipatia26_APIS

# Tercera Prueba HIPATIA 2026
## La Viña Silenciada: recuperar la voz de los sensores

Hace unos días, el sistema de monitorización de la Finca Experimental de Rabanales sufrió un ataque de sabotaje.
Los sensores que vigilan la viña (temperatura, humedad, suelo…) dejaron de enviar datos fiables al Oráculo de la Viña (el broker de contexto).
Sin esos datos, el equipo técnico no puede decidir cuándo regar, cuándo tratar enfermedades ni cuándo vendimiar.

Un antiguo ingeniero dejó un “canal secreto” para volver a activar los sensores.
Pero la clave para activarlo está fragmentada.
Vuestro equipo ha sido elegido para recalibrar los sensores enviando los datos correctos a través de una API y comprobando que el Oráculo vuelve a “hablar”.


En la base de datos de la viña existe un canal oculto para reconectar y recalibrar los sensores cuando el sistema principal falla.
Ese canal es la API:

https://api.atdfiware.uco.es/api/cesens/provision

Vuestro reto es enviar un mensaje especial que identifique vuestro equipo y dos códigos secretos (codigo1 y codigo2).

Si el mensaje está bien construido, el sensor quedará registrado en la base de datos y el Oráculo podrá empezar a guardar sus lecturas.



Solo se activará si el mensaje JSON contiene estos tres campos:

- equipo: nombre del grupo (sin espacios, ej. equipo_lidar)

- codigo1: El código obtenido mediante la prueba NIRS.

- codigo2: El código obtenido mediante la prueba NDVI.

Para llevar a cabo dicha activación deberéis rellenar la siguiente celda con el código correspondiente



In [ ]:
import requests
import json

ENDPOINT_POST = ""

# Construye el payload formado por los atributos equipo, codigo1 y codigo2...
payload = json.dumps({
    "equipo": ""
})

# No hace falta modificar el header
headers = {
  'Content-Type': 'application/json'
}

# Construye la petición
response = requests.request(
    "",                 # TIPO de petición
    ,                   # DIRECCIÓN DE LA PETICIÓN
    headers=headers,
    data=               # IMPORTANTE: mandar el JSON correcto
)

# Si la respuesta es un código 200 quiere decir que ha habido comunicación con la API, pero no quiere decir que la activación se haya realizado con exito...
print(response.text)



Si el Oráculo acepta vuestra ofrenda de datos, registrará el sensor de vuestro equipo en el sistema.
Pero aún no sabéis si realmente está guardando los datos… Para comprobarlo, tendréis que consultar al Oráculo directamente.

El Oráculo de la Viña (Orion) guarda el último estado conocido de cada sensor.
Si vuestro mensaje anterior era correcto, ahora debería existir una entidad asociada a vuestro equipo y códigos.

Solo si sois capaces de leer el último valor usando otra API, demostraréis que el sensor ha despertado…
y el Oráculo os devolverá los últimos dígitos del código del Arca.

Para ello debereis pedir los datos al siguiente endpoint:

https://api.atdfiware.uco.es/api/test/v2/entities/

seguido del nombre de vuestro equipo.

In [ ]:
import requests
import json
equipo = ""
ENDPOINT_GET = ""

ENDPOINT_GET = ENDPOINT_GET + equipo

# ¿Hace falta rellenar el payload?
payload={}

# No hace falta modificar los headers
headers = {
  'Fiware-Service': 'fs_uco',
  'Fiware-ServicePath': '/hiba'
}

# Construye la petición
response = requests.request(
    "",               # Tipo de petición
    ,                 # Endpoint de la petición
    headers=headers,
    data=payload      # ¿Es necesario un payload para pedir datos?
)

# Convertimos la respuesta a dict/list de Python
data = response.json()

# La imprimimos formateada
print(json.dumps(data, indent=3, ensure_ascii=False))

Si habéis llegado hasta aquí, habéis sido capaces de:

- Enviar correctamente una petición POST con vuestro equipo, codigo1 y codigo2.

- Consultar al Oráculo de la Viña mediante una petición GET y leer el último estado de vuestro sensor.

Si todo ha salido bien, el sistema os habrá devuelto un código cifrado.
Ese código contiene los últimos dígitos necesarios para abrir el Arca de la Sostenibilidad.

Guardad ese código como si fuera oro:
es la prueba de que habéis reconectado la viña al mundo de los datos
y la llave final para salvar la cosecha.

Con el siguiente codigo podrás descifrar y obtener el código para abrir el arca.

In [ ]:
def descifra_codigo(data):
    codigo1 = str(data['codigo1']['value']).zfill(2)
    codigo2 = str(data['codigo2']['value'])
    md5_value = data['md5']['value']
    first_md5 = md5_value[0]
    last_md5 = md5_value[-1]

    codigo_unificado = codigo1 + codigo2 + first_md5 + last_md5
    return codigo_unificado

resultado_codigo = descifra_codigo(data)
print(f"El código unificado es: {resultado_codigo}")

In [ ]:
def comprobar_codigo(codigo_a_comprobar):
    parte1_val = int('0x03', 16)
    parte2_val = int('0x43', 16)
    parte3_val = int('0x4B', 16)

    parte1 = str(parte1_val).zfill(2)
    parte2 = str(parte2_val)
    parte3 = str(parte3_val)

    codigo_secreto = parte1 + parte2 + parte3

    if codigo_a_comprobar == codigo_secreto:
        return "¡Enhorabuena! El código es correcto. Has salvado la viña."
    else:
        return "El código no es correcto. Sigue intentándolo."

mensaje_final = comprobar_codigo(resultado_codigo)
print(mensaje_final)